In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Handling Nulls").getOrCreate()

Bookings.csv File

In [ ]:
%%writefile bookings.csv
booking_id,customer_name,city,service_type,provider,booking_amount,booking_status,payment_mode
1001,Aarav Mehta,Hyderabad,Flight,IndiGo,6500,Confirmed,UPI
1002,Sana Khan,Bangalore,Hotel,Pearl Grand,4500,Confirmed,Card
1003,John Mathew,,Flight,Air India,12000,Confirmed,UPI
1004,Ayesha Begum,Hyderabad,Hotel,,7500,Pending,Cash
1005,Vikram Rao,Mumbai,Flight,Vistara,,Confirmed,Card
1006,Divya Sharma,Delhi,Flight,IndiGo,5900,Cancelled,
1007,Imran Ali,Pune,Hotel,Budget Inn,2200,,UPI
1008,Meera Nair,Kochi,Hotel,Hill View Resort,7500,Confirmed,Card
1009,Rohan Das,Kolkata,Flight,Air India,7400,Pending,UPI
1010,Nisha Reddy,Bangalore,Flight,British Airways,62000,Confirmed,Card
1011,Farhan Ali,,Hotel,Skyline Suites,22000,Confirmed,
1012,Neha Singh,Hyderabad,,Emirates,28000,Confirmed,UPI
1013,Arjun Verma,Chennai,Flight,,15000,Cancelled,Cash
1014,Kavya Nair,Mumbai,Hotel,Sea View Stay,,Pending,Card
1015,Ravi Kumar,Delhi,Flight,SpiceJet,4800,Confirmed,UPI

Writing bookings.csv


Exercises on CSV Null Handling


In [ ]:
bookings_df = spark.read.csv("bookings.csv", header=True, inferSchema=True)
bookings_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI |
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card |
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|        UPI |
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|       Cash |
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|       Card |
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|            |
|      100

In [ ]:
bookings_df.printSchema()

root
 |-- booking_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- service_type: string (nullable = true)
 |-- provider: string (nullable = true)
 |-- booking_amount: integer (nullable = true)
 |-- booking_status: string (nullable = true)
 |-- payment_mode: string (nullable = true)



In [ ]:
bookings_df.count()

15

In [ ]:
from pyspark.sql.functions import col
bookings_df.filter(col("city").isNull()).show()

+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|      provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------------+--------------+--------------+------------+
|      1003|  John Mathew|NULL|      Flight|     Air India|         12000|     Confirmed|        UPI |
|      1011|   Farhan Ali|NULL|       Hotel|Skyline Suites|         22000|     Confirmed|            |
+----------+-------------+----+------------+--------------+--------------+--------------+------------+



In [ ]:
bookings_df.filter(col("provider").isNull()).show()

+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+--------+--------------+--------------+------------+
|      1004| Ayesha Begum|Hyderabad|       Hotel|    NULL|          7500|       Pending|       Cash |
|      1013|  Arjun Verma|  Chennai|      Flight|    NULL|         15000|     Cancelled|       Cash |
+----------+-------------+---------+------------+--------+--------------+--------------+------------+



In [ ]:
bookings_df.filter(col("booking_amount").isNull()).show()

+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|booking_id|customer_name|  city|service_type|     provider|booking_amount|booking_status|payment_mode|
+----------+-------------+------+------------+-------------+--------------+--------------+------------+
|      1005|   Vikram Rao|Mumbai|      Flight|      Vistara|          NULL|     Confirmed|       Card |
|      1014|   Kavya Nair|Mumbai|       Hotel|Sea View Stay|          NULL|       Pending|       Card |
+----------+-------------+------+------------+-------------+--------------+--------------+------------+



In [ ]:
bookings_df.filter(col("booking_status").isNull()).show()

+----------+-------------+----+------------+----------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|  provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+----------+--------------+--------------+------------+
|      1007|    Imran Ali|Pune|       Hotel|Budget Inn|          2200|          NULL|        UPI |
+----------+-------------+----+------------+----------+--------------+--------------+------------+



In [ ]:
bookings_df.filter(col("payment_mode").isNull()).show()

+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|booking_id|customer_name| city|service_type|      provider|booking_amount|booking_status|payment_mode|
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+
|      1006| Divya Sharma|Delhi|      Flight|        IndiGo|          5900|     Cancelled|        NULL|
|      1011|   Farhan Ali| NULL|       Hotel|Skyline Suites|         22000|     Confirmed|        NULL|
+----------+-------------+-----+------------+--------------+--------------+--------------+------------+



In [ ]:
from pyspark.sql.functions import when,count
bookings_df.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in bookings_df.columns]
).show()

+----------+-------------+----+------------+--------+--------------+--------------+------------+
|booking_id|customer_name|city|service_type|provider|booking_amount|booking_status|payment_mode|
+----------+-------------+----+------------+--------+--------------+--------------+------------+
|         0|            0|   2|           1|       2|             2|             1|           2|
+----------+-------------+----+------------+--------+--------------+--------------+------------+



In [ ]:
drop_df = bookings_df.na.drop()
drop_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI |
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card |
|      1008|   Meera Nair|    Kochi|       Hotel|Hill View Resort|          7500|     Confirmed|       Card |
|      1009|    Rohan Das|  Kolkata|      Flight|       Air India|          7400|       Pending|        UPI |
|      1010|  Nisha Reddy|Bangalore|      Flight| British Airways|         62000|     Confirmed|       Card |
|      1015|   Ravi Kumar|    Delhi|      Flight|        SpiceJet|          4800|     Confirmed|         UPI|
+---------

In [ ]:
drop_bamt_df = bookings_df.na.drop(subset=["booking_amount"])
drop_bamt_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI |
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card |
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|        UPI |
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|       Cash |
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      1007|    Imran Ali|     Pune|       Hotel|      Budget Inn|          2200|          NULL|        UPI |
|      100

In [ ]:
drop_df = bookings_df.na.drop(subset=["customer_name","service_type","booking_amount"])
drop_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI |
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card |
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|        UPI |
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|       Cash |
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      1007|    Imran Ali|     Pune|       Hotel|      Budget Inn|          2200|          NULL|        UPI |
|      100

In [ ]:
bookings_df.na.fill({"city": "Unknown"}).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI |
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card |
|      1003|  John Mathew|  Unknown|      Flight|       Air India|         12000|     Confirmed|        UPI |
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|       Cash |
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|       Card |
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [ ]:
bookings_df.na.fill({"provider": "Not Available"}).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI |
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card |
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|        UPI |
|      1004| Ayesha Begum|Hyderabad|       Hotel|   Not Available|          7500|       Pending|       Cash |
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|       Card |
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [ ]:
bookings_df.na.fill({"payment_mode": "Not Provided"}).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI |
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card |
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|        UPI |
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|       Cash |
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|       Card |
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|Not Provided|
|      100

In [ ]:
bookings_df.na.fill({"booking_status": "Unknown"}).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI |
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card |
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|        UPI |
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|       Cash |
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|       Card |
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [ ]:
bookings_df.na.fill({"booking_amount": 0}).show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI |
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card |
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|        UPI |
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|       Cash |
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|             0|     Confirmed|       Card |
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|
|      100

In [ ]:
data_quality_df = bookings_df.withColumn(
    "data_quality_status", when(
        col("city").isNull() | col("service_type").isNull() |
        col("provider").isNull() | col("booking_amount").isNull() |
        col("booking_status").isNull(), "Incomplete"
    ).otherwise("Complete"))
data_quality_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI |           Complete|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card |           Complete|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|        UPI |         Incomplete|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|       Cash |         Incomplete|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Conf

In [ ]:
data_quality_df.groupBy("data_quality_status").count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|    7|
|         Incomplete|    8|
+-------------------+-----+



In [ ]:
data_quality_df.filter(col("data_quality_status") == "Complete").show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+-------------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI |           Complete|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card |           Complete|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiGo|          5900|     Cancelled|        NULL|           Complete|
|      1008|   Meera Nair|    Kochi|       Hotel|Hill View Resort|          7500|     Confirmed|       Card |           Complete|
|      1009|    Rohan Das|  Kolkata|      Flight|       Air India|          7400|       Pe

In [ ]:
data_quality_df.filter(col("data_quality_status") == "Incomplete").show()

+----------+-------------+---------+------------+--------------+--------------+--------------+------------+-------------------+
|booking_id|customer_name|     city|service_type|      provider|booking_amount|booking_status|payment_mode|data_quality_status|
+----------+-------------+---------+------------+--------------+--------------+--------------+------------+-------------------+
|      1003|  John Mathew|     NULL|      Flight|     Air India|         12000|     Confirmed|        UPI |         Incomplete|
|      1004| Ayesha Begum|Hyderabad|       Hotel|          NULL|          7500|       Pending|       Cash |         Incomplete|
|      1005|   Vikram Rao|   Mumbai|      Flight|       Vistara|          NULL|     Confirmed|       Card |         Incomplete|
|      1007|    Imran Ali|     Pune|       Hotel|    Budget Inn|          2200|          NULL|        UPI |         Incomplete|
|      1011|   Farhan Ali|     NULL|       Hotel|Skyline Suites|         22000|     Confirmed|        NU

In [ ]:
tax_df = bookings_df.withColumn("tax",col("booking_amount")*0.05)
tax_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|   tax|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI | 325.0|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card | 225.0|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|        UPI | 600.0|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|       Cash | 375.0|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Confirmed|       Card |  NULL|
|      1006| Divya Sharma|    Delhi|      Flight|          IndiG

In [ ]:
final_df = tax_df.withColumn("final_amount",col("booking_amount")+col("tax"))
final_df.show()

+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+------------+
|booking_id|customer_name|     city|service_type|        provider|booking_amount|booking_status|payment_mode|   tax|final_amount|
+----------+-------------+---------+------------+----------------+--------------+--------------+------------+------+------------+
|      1001|  Aarav Mehta|Hyderabad|      Flight|          IndiGo|          6500|     Confirmed|        UPI | 325.0|      6825.0|
|      1002|    Sana Khan|Bangalore|       Hotel|     Pearl Grand|          4500|     Confirmed|       Card | 225.0|      4725.0|
|      1003|  John Mathew|     NULL|      Flight|       Air India|         12000|     Confirmed|        UPI | 600.0|     12600.0|
|      1004| Ayesha Begum|Hyderabad|       Hotel|            NULL|          7500|       Pending|       Cash | 375.0|      7875.0|
|      1005|   Vikram Rao|   Mumbai|      Flight|         Vistara|          NULL|     Conf

In [ ]:
from pyspark.sql import functions as F
bookings_df.filter(col("booking_status") == "Confirmed") \
  .agg(F.sum("booking_amount").alias("Revenue")).show()

+-------+
|Revenue|
+-------+
| 147300|
+-------+



In [ ]:
bookings_df.na.fill({"service_type": "Unknown"}).groupBy("service_type").count().show()

+------------+-----+
|service_type|count|
+------------+-----+
|     Unknown|    1|
|       Hotel|    6|
|      Flight|    8|
+------------+-----+



In [ ]:
bookings_df.na.fill({"city": "Unknown"}).groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    2|
|    Kochi|    1|
|  Chennai|    1|
|   Mumbai|    2|
|  Kolkata|    1|
|  Unknown|    2|
|     Pune|    1|
|    Delhi|    2|
|Hyderabad|    3|
+---------+-----+



In [ ]:
bookings_df.na.fill({"booking_amount": 0}) \
  .agg(F.avg("booking_amount").alias("Average Booking Amount")).show()

+----------------------+
|Average Booking Amount|
+----------------------+
|    12353.333333333334|
+----------------------+



In [ ]:
bookings_df.na.drop(subset=["booking_amount"]) \
  .agg(F.avg("booking_amount").alias("Average Booking Amount")).show()

+----------------------+
|Average Booking Amount|
+----------------------+
|    14253.846153846154|
+----------------------+



In [ ]:
bookings_df.na.fill({"booking_amount": 0}) \
  .agg(F.avg("booking_amount").alias("Average Booking Amount With 0")).show()
bookings_df.na.drop(subset=["booking_amount"]) \
  .agg(F.avg("booking_amount").alias("Average Booking Amount without Null")).show()

+-----------------------------+
|Average Booking Amount With 0|
+-----------------------------+
|           12353.333333333334|
+-----------------------------+

+-----------------------------------+
|Average Booking Amount without Null|
+-----------------------------------+
|                 14253.846153846154|
+-----------------------------------+



In [ ]:
clean_df = bookings_df.na.fill({
    "city": "Unknown",
    "provider": "Not Available",
    "payment_mode": "Not Provided",
    "booking_status": "Unknown",
    "booking_amount": 0
})
clean_df.write.mode("overwrite").parquet("clean_bookings.parquet")

JSON Dataset with Nulls: customers.json

In [ ]:
%%writefile customers.json
[
  {
    "customer_id": 1,
    "name": "Aarav Mehta",
    "city": "Hyderabad",
    "membership": "Gold",
    "contact": {
      "phone": "9876500011",
      "email": "aarav@mail.com"
    },
    "preferences": {
      "preferred_service": "Flight",
      "budget_range": "Medium"
    }
  },
  {
    "customer_id": 2,
    "name": "Sana Khan",
    "city": "Bangalore",
    "membership": "Silver",
    "contact": {
      "phone": null,
      "email": "sana@mail.com"
    },
    "preferences": {
      "preferred_service": "Hotel",
      "budget_range": null
    }
  },
  {
    "customer_id": 3,
    "name": "John Mathew",
    "city": null,
    "membership": "Gold",
    "contact": {
      "phone": "9876500013",
      "email": null
    },
    "preferences": {
      "preferred_service": "Flight",
      "budget_range": "High"
    }
  },
  {
    "customer_id": 4,
    "name": "Ayesha Begum",
    "city": "Hyderabad",
    "membership": null,
    "contact": {
      "phone": "9876500014",
      "email": "ayesha@mail.com"
    },
    "preferences": {
      "preferred_service": null,
      "budget_range": "Low"
    }
  },
  {
    "customer_id": 5,
    "name": "Vikram Rao",
    "city": "Mumbai",
    "membership": "Platinum",
    "contact": {
      "phone": null,
      "email": null
    },
    "preferences": {
      "preferred_service": "Flight",
      "budget_range": "High"
    }
  }
]

Writing customers.json


In [ ]:
# Read JSON:
customers_df = spark.read.option("multiline","true").json("customers.json")
customers_df.show(truncate=False)
customers_df.printSchema()

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: strin

Flatten JSON

In [ ]:
flat_customers_df = customers_df.select(
    "customer_id",
    "name",
    "city",
    "membership",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    col("preferences.preferred_service").alias("preferred_service"),
    col("preferences.budget_range").alias("budget_range"))
flat_customers_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



JSON Exercises


In [ ]:
customers_df.show()

+---------+--------------------+-----------+----------+------------+----------------+
|     city|             contact|customer_id|membership|        name|     preferences|
+---------+--------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, ...|          1|      Gold| Aarav Mehta|{Medium, Flight}|
|Bangalore|{sana@mail.com, N...|          2|    Silver|   Sana Khan|   {NULL, Hotel}|
|     NULL|  {NULL, 9876500013}|          3|      Gold| John Mathew|  {High, Flight}|
|Hyderabad|{ayesha@mail.com,...|          4|      NULL|Ayesha Begum|     {Low, NULL}|
|   Mumbai|        {NULL, NULL}|          5|  Platinum|  Vikram Rao|  {High, Flight}|
+---------+--------------------+-----------+----------+------------+----------------+



In [ ]:
customers_df.printSchema()

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- membership: string (nullable = true)
 |-- name: string (nullable = true)
 |-- preferences: struct (nullable = true)
 |    |-- budget_range: string (nullable = true)
 |    |-- preferred_service: string (nullable = true)



In [ ]:
customers_df.show(truncate=False)

+---------+-----------------------------+-----------+----------+------------+----------------+
|city     |contact                      |customer_id|membership|name        |preferences     |
+---------+-----------------------------+-----------+----------+------------+----------------+
|Hyderabad|{aarav@mail.com, 9876500011} |1          |Gold      |Aarav Mehta |{Medium, Flight}|
|Bangalore|{sana@mail.com, NULL}        |2          |Silver    |Sana Khan   |{NULL, Hotel}   |
|NULL     |{NULL, 9876500013}           |3          |Gold      |John Mathew |{High, Flight}  |
|Hyderabad|{ayesha@mail.com, 9876500014}|4          |NULL      |Ayesha Begum|{Low, NULL}     |
|Mumbai   |{NULL, NULL}                 |5          |Platinum  |Vikram Rao  |{High, Flight}  |
+---------+-----------------------------+-----------+----------+------------+----------------+



In [ ]:
customers_df.select(
    "customer_id",
    "name",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
).show()

+-----------+------------+----------+---------------+
|customer_id|        name|     phone|          email|
+-----------+------------+----------+---------------+
|          1| Aarav Mehta|9876500011| aarav@mail.com|
|          2|   Sana Khan|      NULL|  sana@mail.com|
|          3| John Mathew|9876500013|           NULL|
|          4|Ayesha Begum|9876500014|ayesha@mail.com|
|          5|  Vikram Rao|      NULL|           NULL|
+-----------+------------+----------+---------------+



In [ ]:
customers_df.select(
    "customer_id",
    "name",
    col("preferences.preferred_service").alias("preferred_service"),
    col("preferences.budget_range").alias("budget_range")
).show()

+-----------+------------+-----------------+------------+
|customer_id|        name|preferred_service|budget_range|
+-----------+------------+-----------------+------------+
|          1| Aarav Mehta|           Flight|      Medium|
|          2|   Sana Khan|            Hotel|        NULL|
|          3| John Mathew|           Flight|        High|
|          4|Ayesha Begum|             NULL|         Low|
|          5|  Vikram Rao|           Flight|        High|
+-----------+------------+-----------------+------------+



In [ ]:
customers_df.select("name", "city", "contact.phone", "contact.email").show()

+------------+---------+----------+---------------+
|        name|     city|     phone|          email|
+------------+---------+----------+---------------+
| Aarav Mehta|Hyderabad|9876500011| aarav@mail.com|
|   Sana Khan|Bangalore|      NULL|  sana@mail.com|
| John Mathew|     NULL|9876500013|           NULL|
|Ayesha Begum|Hyderabad|9876500014|ayesha@mail.com|
|  Vikram Rao|   Mumbai|      NULL|           NULL|
+------------+---------+----------+---------------+



In [ ]:
customers_df.filter(col("city").isNull()).show()

+----+------------------+-----------+----------+-----------+--------------+
|city|           contact|customer_id|membership|       name|   preferences|
+----+------------------+-----------+----------+-----------+--------------+
|NULL|{NULL, 9876500013}|          3|      Gold|John Mathew|{High, Flight}|
+----+------------------+-----------+----------+-----------+--------------+



In [ ]:
customers_df.filter(col("contact.phone").isNull()).show()

+---------+--------------------+-----------+----------+----------+--------------+
|     city|             contact|customer_id|membership|      name|   preferences|
+---------+--------------------+-----------+----------+----------+--------------+
|Bangalore|{sana@mail.com, N...|          2|    Silver| Sana Khan| {NULL, Hotel}|
|   Mumbai|        {NULL, NULL}|          5|  Platinum|Vikram Rao|{High, Flight}|
+---------+--------------------+-----------+----------+----------+--------------+



In [ ]:
customers_df.filter(col("contact.email").isNull()).show()

+------+------------------+-----------+----------+-----------+--------------+
|  city|           contact|customer_id|membership|       name|   preferences|
+------+------------------+-----------+----------+-----------+--------------+
|  NULL|{NULL, 9876500013}|          3|      Gold|John Mathew|{High, Flight}|
|Mumbai|      {NULL, NULL}|          5|  Platinum| Vikram Rao|{High, Flight}|
+------+------------------+-----------+----------+-----------+--------------+



In [ ]:
customers_df.filter(col("membership").isNull()).show()

+---------+--------------------+-----------+----------+------------+-----------+
|     city|             contact|customer_id|membership|        name|preferences|
+---------+--------------------+-----------+----------+------------+-----------+
|Hyderabad|{ayesha@mail.com,...|          4|      NULL|Ayesha Begum|{Low, NULL}|
+---------+--------------------+-----------+----------+------------+-----------+



In [ ]:
customers_df.filter(col("preferences.preferred_service").isNull()).show()

+---------+--------------------+-----------+----------+------------+-----------+
|     city|             contact|customer_id|membership|        name|preferences|
+---------+--------------------+-----------+----------+------------+-----------+
|Hyderabad|{ayesha@mail.com,...|          4|      NULL|Ayesha Begum|{Low, NULL}|
+---------+--------------------+-----------+----------+------------+-----------+



In [ ]:
customers_df.filter(col("preferences.budget_range").isNull()).show()

+---------+--------------------+-----------+----------+---------+-------------+
|     city|             contact|customer_id|membership|     name|  preferences|
+---------+--------------------+-----------+----------+---------+-------------+
|Bangalore|{sana@mail.com, N...|          2|    Silver|Sana Khan|{NULL, Hotel}|
+---------+--------------------+-----------+----------+---------+-------------+



In [ ]:
from pyspark.sql.functions import when,count,col
flat_customers_df.select([
    count(when(col(c).isNull(), c)).alias(c) for c in flat_customers_df.columns
]).show()

+-----------+----+----+----------+-----+-----+-----------------+------------+
|customer_id|name|city|membership|phone|email|preferred_service|budget_range|
+-----------+----+----+----------+-----+-----+-----------------+------------+
|          0|   0|   1|         1|    2|    2|                1|           1|
+-----------+----+----+----------+-----+-----+-----------------+------------+



In [ ]:
flat_customers_df.na.fill({"city": "Unknown"}).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|  Unknown|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [ ]:
flat_customers_df.na.fill({"membership": "Standard"}).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|  Standard|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [ ]:
flat_customers_df.na.fill({"phone": "Not Provided"}).show()

+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|       phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|  9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|Not Provided|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|  9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|  9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|Not Provided|           NULL|           Flight|        High|
+-----------+------------+---------+----------+------------+---------------+-----------------+------------+



In [ ]:
flat_customers_df.na.fill({"email": "Not Provided"}).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|   Not Provided|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|   Not Provided|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [ ]:
flat_customers_df.na.fill({"preferred_service": "Not Selected"}).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|     Not Selected|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [ ]:
flat_customers_df.na.fill({"budget_range": "Unknown"}).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|     Unknown|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+



In [ ]:
customer_quality_df = flat_customers_df.withColumn(
    "customer_quality_status", when(
        col("city").isNull() | col("phone").isNull() | col("email").isNull() |
        col("membership").isNull() | col("preferred_service").isNull(),
        "Incomplete"
    ).otherwise("Complete"))
customer_quality_df.show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|          1| Aarav Mehta|Hyderabad|      Gold|9876500011| aarav@mail.com|           Flight|      Medium|               Complete|
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|             Incomplete|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|             Incomplete|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|             Incomplete|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Fligh

In [ ]:
customer_quality_df.groupBy("customer_quality_status").count().show()

+-----------------------+-----+
|customer_quality_status|count|
+-----------------------+-----+
|               Complete|    1|
|             Incomplete|    4|
+-----------------------+-----+



In [ ]:
customer_quality_df.filter(col("customer_quality_status") == "Complete").show()

+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|customer_id|       name|     city|membership|     phone|         email|preferred_service|budget_range|customer_quality_status|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+
|          1|Aarav Mehta|Hyderabad|      Gold|9876500011|aarav@mail.com|           Flight|      Medium|               Complete|
+-----------+-----------+---------+----------+----------+--------------+-----------------+------------+-----------------------+



In [ ]:
customer_quality_df.filter(col("customer_quality_status") == "Incomplete").show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|customer_quality_status|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+-----------------------+
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|             Incomplete|
|          3| John Mathew|     NULL|      Gold|9876500013|           NULL|           Flight|        High|             Incomplete|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|             Incomplete|
|          5|  Vikram Rao|   Mumbai|  Platinum|      NULL|           NULL|           Flight|        High|             Incomplete|
+-----------+------------+---------+----------+----------+---------------+----------------

In [ ]:
flat_customers_df.na.fill({"membership": "Standard"}).groupBy("membership").count().show()

+----------+-----+
|membership|count|
+----------+-----+
|  Platinum|    1|
|    Silver|    1|
|      Gold|    2|
|  Standard|    1|
+----------+-----+



In [ ]:
flat_customers_df.na.fill({"preferred_service": "Not Selected"}).groupBy("preferred_service").count().show()

+-----------------+-----+
|preferred_service|count|
+-----------------+-----+
|     Not Selected|    1|
|            Hotel|    1|
|           Flight|    3|
+-----------------+-----+



In [ ]:
flat_customers_df.write.mode("overwrite").parquet("customers_flat.parquet")

In [ ]:
clean_df = flat_customers_df.na.fill({
    "city": "Unknown",
    "membership": "Standard",
    "phone": "Not Provided",
    "email": "Not Provided",
    "preferred_service": "Not Selected",
    "budget_range": "Unknown"
})
clean_df.write.mode("overwrite").parquet("clean_customers.parquet")

In [ ]:
print("Original Record Count:",flat_customers_df.count())
print("Clean Record Count:",clean_df.count())

Original Record Count: 5
Clean Record Count: 5


In [ ]:
flat_customers_df.filter(col("phone").isNull() | col("email").isNull()).show()

+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|customer_id|       name|     city|membership|     phone|        email|preferred_service|budget_range|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+
|          2|  Sana Khan|Bangalore|    Silver|      NULL|sana@mail.com|            Hotel|        NULL|
|          3|John Mathew|     NULL|      Gold|9876500013|         NULL|           Flight|        High|
|          5| Vikram Rao|   Mumbai|  Platinum|      NULL|         NULL|           Flight|        High|
+-----------+-----------+---------+----------+----------+-------------+-----------------+------------+



In [ ]:
flat_customers_df.filter(
    col("preferred_service").isNull() | col("budget_range").isNull()
).show()

+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|customer_id|        name|     city|membership|     phone|          email|preferred_service|budget_range|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+
|          2|   Sana Khan|Bangalore|    Silver|      NULL|  sana@mail.com|            Hotel|        NULL|
|          4|Ayesha Begum|Hyderabad|      NULL|9876500014|ayesha@mail.com|             NULL|         Low|
+-----------+------------+---------+----------+----------+---------------+-----------------+------------+

